# 01 — Data Audit

Grain, schema, and the question that matters most for a demand panel: **is a zero a real zero?**

A zero can mean the product was on shelf and nobody bought it, or that it did not exist yet, or that it was delisted. Training on all three as the same thing teaches the model that demand collapses at the start and end of a product's life — an artefact of the panel's shape.

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
warnings.filterwarnings('ignore')

from src.config import load_config
from src.data.loader import load_panel
cfg = load_config()
panel = load_panel(cfg)
print(f'panel: {len(panel):,} rows, {panel.series_id.nunique()} series, {panel.date.nunique()} days')

panel: 1,003,600 rows, 600 series, 1969 days


## Grain and schema

In [2]:
from src.data.schema import DATA_DICTIONARY
import pandas as pd
print('grain:', cfg['data']['grain'], '| target:', cfg.target)
pd.DataFrame([{'column': s.name, 'kind': s.kind, 'source': s.source,
               'description': s.description, 'notes': s.notes}
              for s in DATA_DICTIONARY])

grain: ['sku_id', 'store_id', 'date'] | target: demand


,column,kind,source,description,notes
0,series_id,key,target/static/temporal,SKU x store series identifier.,"The forecasting grain. 3,049 SKUs x 10 stores ..."
1,date,key,target,Calendar day.,"Daily frequency, contiguous within each series."
2,demand,target,target,Units sold that day.,"Integer, non-negative. ~60% of observations ar..."
3,sku_id,categorical,static,"Product identifier, shared across stores.",
4,dept_id,categorical,static,Department (7 values).,
5,cat_id,categorical,static,Category: FOODS / HOUSEHOLD / HOBBIES.,
6,store_id,categorical,static,Store identifier.,
7,state_id,categorical,static,State: CA / TX / WI.,Determines which SNAP calendar applies.
8,sell_price,numeric,temporal,Unit selling price for that series that week.,No missing values in the mirror; the mirror tr...
9,snap,binary,temporal,1 if that date is a SNAP benefit day in the se...,The dataset's only genuine promotion-like sign...


## Structural integrity and the zero taxonomy

In [3]:
from src.data.audit import zero_taxonomy, lifecycle, calendar_gaps, spikes
print('duplicate (series,date):', int(panel.duplicated(['series_id','date']).sum()))
print('series with internal gaps:', calendar_gaps(panel))
print('negative demand:', int((panel.demand < 0).sum()))
zero_taxonomy(panel)

duplicate (series,date): 0
series with internal gaps: 0
negative demand: 0


{'n_zero': 591075,
 'zero_share': 0.5889547628537266,
 'n_pre_launch_zero': 0,
 'n_post_discontinuation_zero': 1931,
 'n_true_zero': 589144,
 'true_zero_share_of_zeros': 0.9967330710992682}

In [4]:
lifecycle(panel)

{'n_series': 600,
 'n_launched_late': 435,
 'n_distinct_start_dates': 242,
 'n_discontinued': 0,
 'median_history_days': 1952.5,
 'min_history_days': 733.0}

In [5]:
spikes(panel)

{'n_spike_days': 13, 'n_spike_series': 8}

## Intermittency: the fact that shapes every later choice

In [6]:
from src.data.loader import demand_stats, classify_demand
st = demand_stats(panel); st['demand_class'] = classify_demand(st)
print(st.demand_class.value_counts())
st[['ADI','CV2','zero_share','mean_demand']].describe().round(3)

demand_class
Intermittent    449
Lumpy           103
Smooth           31
Erratic          17
Name: count, dtype: int64


,ADI,CV2,zero_share,mean_demand
count,600.000,600.000,600.000,600.000
mean,3.926,0.404,0.593,1.527
std,3.687,0.809,0.228,3.350
min,1.003,0.031,0.003,0.040
25%,1.741,0.239,0.426,0.306
50%,2.646,0.343,0.622,0.628
75%,4.581,0.450,0.782,1.298
max,27.667,19.544,0.964,39.845


## The full audit, including the executable leakage probe

The probe rebuilds features after overwriting every actual past the forecast origin. Any feature that changes was reading the future.

In [7]:
from src.data import audit
res = audit.run(cfg)

panel                  : 1,003,600 rows, 600 series, 1,969 days (2011-01-29..2016-06-19)
schema ok              : True
duplicates / gaps      : 0 / 0
negative demand        : 0
missing values         : none
invalid ranges         : none
zero share             : 58.90%
  pre-launch zeros     : 0
  post-delist zeros    : 1,931
  TRUE zeros           : 589,144 (99.7% of zeros)
launched late / delisted: 435 / 0
spikes                 : 13 days in 8 series
demand classes         : {'Intermittent': 449, 'Lumpy': 103, 'Smooth': 31, 'Erratic': 17}
intermittent+lumpy     : 92.0%
LEAKAGE probe          : leak_free=True (54 features, 1,800 rows)

[written] /projects/sandbox/aiml-portfolio/aiml4-demand-forecasting/reports/data_audit.md


In [8]:
res.leakage_checks

{'fold': 'fold_4',
 'origin': '2016-04-24',
 'n_features_checked': 54,
 'n_rows_compared': 1800,
 'features_changed_by_future_mutation': [],
 'leak_free': True}